In [1]:
# =============================================================================
# Article Analysis Notebook - Boeing Air India Crash
# =============================================================================

import requests
from bs4 import BeautifulSoup
import re
import json
from datetime import datetime

# For LLM simulation (no API key needed)
# We'll create rule-based + simple heuristic functions as "LLM"
# For Deep Learning: Use Hugging Face transformers (VADER + simple BERT-like via pipeline)

print("📥 Fetching article...")
url = "https://apnews.com/article/boeing-aviation-aircraft-air-india-crash-f12b20e65dc57ae655a1e0759b58938f"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.get(url, headers=headers, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

# Extract main article text
article_text = ""
for p in soup.find_all(['p', 'h1', 'h2', 'h3']):
    text = p.get_text(strip=True)
    if len(text) > 30 and not text.startswith("TOP STORIES"):
        article_text += text + "\n\n"

# Clean text
article_text = re.sub(r'\s+', ' ', article_text).strip()
print("✅ Article fetched successfully!\n")
print("Article Length:", len(article_text), "characters\n")
print("="*80)
print(article_text[:800] + "...\n")  # Preview

📥 Fetching article...
✅ Article fetched successfully!

Article Length: 3321 characters

Copyright 2026 The Associated Press. All Rights Reserved. Copyright 2026 The Associated Press. All Rights Reserved. A look at Boeing’s recent troubles after Air India crash The Boeing logo is displayed at the company’s factory, Sept. 24, 2024, in Renton, Wash. (AP Photo/Lindsey Wasson, File) ▶ Follow live updates onthe Air India plane crash. The crash of aBoeing 787 passenger jet in Indiaminutes after takeoff on Thursday is putting the spotlight back on a beleaguered manufacturer though it was not immediately clear why the plane crashed. The Air India 787 went down in the northwestern city of Ahmedabad with more than 240 people aboard shortly after takeoff, authorities said. It was the first fatal crash since the plane, also known as the Dreamliner, went into service in 2011, according to ...



In [2]:
# =============================================================================
# 1. Sentiment, Intent, Emotions Analysis
# =============================================================================

def rule_based_sentiment(text):
    """Simple rule-based sentiment (LLM simulation)"""
    negative_words = ['crash', 'troubles', 'loss', 'deadly', 'killed', 'grounded', 'problems', 'faulty', 'struggle']
    positive_words = ['progress', 'orders', 'big', 'secured']
    
    neg_count = sum(1 for word in negative_words if word in text.lower())
    pos_count = sum(1 for word in positive_words if word in text.lower())
    
    if neg_count > pos_count + 2:
        return "Negative", 0.85
    elif pos_count > neg_count:
        return "Positive", 0.6
    else:
        return "Neutral", 0.7

def detect_intent(text):
    """Detect primary intent"""
    if any(word in text.lower() for word in ['troubles', 'problems', 'crash', 'killed']):
        return "Informative / Highlighting Risks"
    return "Neutral Reporting"

def detect_emotions(text):
    """Simple emotion detection"""
    emotions = {
        "Fear/Anxiety": "High" if "crash" in text.lower() else "Low",
        "Sadness": "High" if any(w in text.lower() for w in ["killed", "deadly", "loss"]) else "Medium",
        "Anger": "Medium",
        "Surprise": "Medium"
    }
    return emotions

print("🔍 ANALYSIS 1: Rule-based LLM Simulation")
sentiment, score = rule_based_sentiment(article_text)
print(f"Sentiment: {sentiment} (confidence: {score:.2f})")
print(f"Intent: {detect_intent(article_text)}")
print("Emotions:", detect_emotions(article_text))

🔍 ANALYSIS 1: Rule-based LLM Simulation
Sentiment: Negative (confidence: 0.85)
Intent: Informative / Highlighting Risks
Emotions: {'Fear/Anxiety': 'High', 'Sadness': 'High', 'Anger': 'Medium', 'Surprise': 'Medium'}


In [3]:
# Deep Learning Method: Using Hugging Face (VADER + transformers)
# =============================================================================

# Install required packages including PyTorch
!pip install -q vaderSentiment transformers torch

# Import statements with error handling for torch
try:
    import torch
    print("✅ PyTorch imported successfully")
except ImportError:
    print("❌ PyTorch not found. Please restart the kernel and run this cell again.")
    # Alternative: try installing torch with specific index
    !pip install torch --index-url https://download.pytorch.org/whl/cpu
    import torch

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline

print("\n🔬 Deep Learning Analysis (VADER + DistilBERT)")

# Note: Make sure 'article_text' is defined before running this code
# If not defined, uncomment and modify the line below:
# article_text = "Your sample text here for sentiment analysis"

# VADER
analyzer = SentimentIntensityAnalyzer()
vader_scores = analyzer.polarity_scores(article_text)
print("VADER Scores:", vader_scores)

# Emotion classification using pipeline
emotion_classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None)
emotions_dl = emotion_classifier(article_text[:1000])[0]  # Truncate for speed

print("\nTop Emotions (Deep Learning):")
for e in sorted(emotions_dl, key=lambda x: x['score'], reverse=True)[:4]:
    print(f"  {e['label']}: {e['score']:.4f}")

✅ PyTorch imported successfully

🔬 Deep Learning Analysis (VADER + DistilBERT)
VADER Scores: {'neg': 0.1, 'neu': 0.862, 'pos': 0.037, 'compound': -0.9883}


pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

C:\Users\pudar\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pudar\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


Top Emotions (Deep Learning):
  sadness: 0.9662
  fear: 0.0091
  surprise: 0.0089
  anger: 0.0068


In [4]:
# =============================================================================
# 2. Topic Proportions (Technology, Aviation, Policies)
# =============================================================================

def topic_percentage(text):
    aviation_keywords = ['boeing', '787', 'dreamliner', 'plane', 'jet', 'aircraft', 'flight', 'crash', 'takeoff']
    tech_keywords = ['battery', 'lithium', 'software', 'sensor', '737 max']
    policy_keywords = ['justice department', 'regulators', 'grounded', 'investigation', 'scrutiny', 'strike']
    
    words = text.lower().split()
    total = len(words)
    
    av = sum(1 for w in words if any(k in w for k in aviation_keywords)) / total * 100
    tech = sum(1 for w in words if any(k in w for k in tech_keywords)) / total * 100
    pol = sum(1 for w in words if any(k in w for k in policy_keywords)) / total * 100
    
    return {
        "Aviation": round(av, 1),
        "Technology": round(tech, 1),
        "Policies/Regulation": round(pol, 1)
    }

print("\n📊 TOPIC DISTRIBUTION")
topics = topic_percentage(article_text)
for topic, pct in topics.items():
    print(f"{topic}: {pct}%")


📊 TOPIC DISTRIBUTION
Aviation: 9.7%
Technology: 0.6%
Policies/Regulation: 1.3%


In [5]:
# =============================================================================
# 3. Comparison
# =============================================================================

print("\n" + "="*80)
print("📋 FINAL COMPARISON")
print("="*80)

print("Rule-based / Simulated LLM:")
print(f"• Sentiment : Negative")
print(f"• Intent    : Informative / Highlighting Boeing's ongoing issues")
print(f"• Emotions  : Fear, Sadness dominant")

print("\nDeep Learning (VADER + DistilRoBERTa):")
print(f"• Sentiment : Negative (compound score ~ -0.XX)")
print(f"• Emotions  : sadness, fear, anger (top scores)")

print("\nTopic Coverage:")
print(json.dumps(topics, indent=2))

print("\n✅ Analysis Complete!")
print("The article is primarily negative in tone, focused heavily on aviation safety and Boeing's history of issues.")


📋 FINAL COMPARISON
Rule-based / Simulated LLM:
• Sentiment : Negative
• Intent    : Informative / Highlighting Boeing's ongoing issues
• Emotions  : Fear, Sadness dominant

Deep Learning (VADER + DistilRoBERTa):
• Sentiment : Negative (compound score ~ -0.XX)
• Emotions  : sadness, fear, anger (top scores)

Topic Coverage:
{
  "Aviation": 9.7,
  "Technology": 0.6,
  "Policies/Regulation": 1.3
}

✅ Analysis Complete!
The article is primarily negative in tone, focused heavily on aviation safety and Boeing's history of issues.
